In [1]:
'ciao'

'ciao'

In [2]:
import os
import os.path as osp
if osp.split(os.getcwd())[-1] == 'notebooks':
    os.chdir('..')

os.getcwd()

'e:\\Edoardo\\Education\\SportDataCampus\\ScoutingAgent'

In [3]:
from wyscout import get_match_events, get_match_details

# Start

In [4]:
events = get_match_events(5718124)['events']

In [5]:
def get_possession_events(events: list[dict], possession_id: int) -> list[dict]:
    """
    Extracts possession events from a list of Wyscout events.

    Args:
        events (list[dict]): A list of dictionaries containing Wyscout event data.
    """

    possession_events = list(
        filter(
            lambda event: event.get('possession') is not None and event['possession'].get('id') == 101,
            events
        )
    )
    return possession_events

In [6]:
def get_match_possession(events: list[dict]) -> list[dict]:

    """
    Returns a list of all unique possessions in the match as lists of event dicts (ordered as in events).

    Args:
        events (list[dict]): A list of dictionaries containing Wyscout event data for the match.

    Returns:
        list[dict]: A list where each item is a dict:
            {
                "possession_id": int,
                "events": list[dict]
            }
    """
    from collections import defaultdict

    # Group events by possession id
    possessions = defaultdict(list)
    for event in events:
        pid = None
        if event.get('possession') and event['possession'].get('id') is not None:
            pid = event['possession']['id']
        if pid is not None:
            possessions[pid].append(event)

    # Result: list of dicts with possession_id and events
    return [
        {
            "possession_id": pid,
            "events": possession_events
        }
        for pid, possession_events in sorted(possessions.items())
    ]

In [7]:
possessions = get_match_possession(events)

## Possession analysis

In [8]:
pid = possessions[0]['possession_id']
possession_events = possessions[0]['events']
match_info = get_match_details(5718124, details=['teams'])

In [9]:
from possession_analyzer import analyze_possession

possession_analysis = analyze_possession(possession_events, match_info, events)

In [10]:
match_info['teamsData']['692']['team'].keys()

dict_keys(['wyId', 'name', 'officialName', 'city', 'area', 'type', 'category', 'gender', 'children', 'imageDataURL'])

In [11]:
possession_analysis

{'possession_id': 2927832885,
 'num_events': 4,
 'team_in_possession': 675,
 'team_in_possession_name': 'Real Madrid',
 'opponent_team_name': 'Celta de Vigo',
 'opponent_team_id': 692,
 'duration': 1.48391,
 'pass_count': 0,
 'avg_pass_speed': 0.0,
 'total_x_advancement': -3.0,
 'time_in_thirds': {'defensive': 100.0, 'middle': 0.0, 'attacking': 0.0},
 'ball_circulation_count': 0,
 'match_state': {'home_score': 0, 'away_score': 0, 'leading_team': None},
 'players_involved': [{'id': 276979, 'name': 'F. Mendy', 'team_id': 675},
  {'id': 569212, 'name': 'Ferrán Jutglà', 'team_id': 692}],
 'temporal_moment': {'period': '1H', 'matchTimestamp': '00:00:07.268'},
 'preceding_events': [{'id': 2927833236,
   'matchId': 5718124,
   'matchPeriod': '1H',
   'minute': 0,
   'second': 4,
   'matchTimestamp': '00:00:04.866',
   'videoTimestamp': '5.866453',
   'relatedEventId': 2927833238,
   'type': {'primary': 'pass',
    'secondary': ['forward_pass',
     'long_pass',
     'loss',
     'pass_to_fina

## Vertex AI

In [12]:
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_APPLICATION_CREDENTIALS"]

'.secrets/vertexai-wyscout.json'

In [13]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(
    project=os.environ["GCP_PROJECT_ID"],
    location=os.environ["VERTEX_LOCATION"],
)
model = GenerativeModel(os.environ["VERTEX_MODEL_NAME"])

e:\Edoardo\Education\SportDataCampus\ScoutingAgent\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [14]:
resp = model.generate_content("Rispondi in una riga: che modello sei?")
resp

candidates {
  content {
    role: "model"
    parts {
      text: "Sono un modello linguistico di grandi dimensioni, addestrato da Google."
    }
  }
  finish_reason: STOP
  avg_logprobs: -0.00032898519809047383
}
usage_metadata {
  prompt_token_count: 12
  candidates_token_count: 15
  total_token_count: 27
  prompt_tokens_details {
    modality: TEXT
    token_count: 12
  }
  candidates_tokens_details {
    modality: TEXT
    token_count: 15
  }
}
model_version: "gemini-2.5-flash-lite"
create_time {
  seconds: 1774187481
  nanos: 315760000
}
response_id: "2fO_afCiE5-rz_IPoeWasQY"

In [15]:
from possession_description import render_general_possession_prompt

In [16]:
possession_events

[{'id': 2927832885,
  'matchId': 5718124,
  'matchPeriod': '1H',
  'minute': 0,
  'second': 7,
  'matchTimestamp': '00:00:07.268',
  'videoTimestamp': '8.268542',
  'relatedEventId': 2927832886,
  'type': {'primary': 'duel',
   'secondary': ['loose_ball_duel', 'recovery', 'counterpressing_recovery']},
  'location': {'x': 30, 'y': 3},
  'team': {'id': 675, 'name': 'Real Madrid', 'formation': '5-4-1'},
  'opponentTeam': {'id': 692, 'name': 'Celta de Vigo', 'formation': '3-4-3'},
  'player': {'id': 276979, 'name': 'F. Mendy', 'position': 'LB5'},
  'pass': None,
  'shot': None,
  'groundDuel': None,
  'aerialDuel': None,
  'infraction': None,
  'carry': None,
  'possession': {'id': 2927832885,
   'duration': '1.48391',
   'types': [],
   'eventsNumber': 4,
   'eventIndex': 0,
   'startLocation': {'x': 30, 'y': 3},
   'endLocation': {'x': 27, 'y': 0},
   'team': {'id': 675, 'name': 'Real Madrid', 'formation': '5-4-1'},
   'attack': None}},
 {'id': 2927833238,
  'matchId': 5718124,
  'matchP

In [17]:
prompt = render_general_possession_prompt(possession_analysis)
print(prompt)

You are a football match analyst of an elite football team. Analyze the following possession sequence and provide a detailed, natural language description of the overall possession.

IMPORTANT CONTEXT:
- The team in possession attacks from LEFT to RIGHT (x=0 is their defensive goal, x=100 is their attacking goal)
- The field is divided into 9 zones using juego de posicion (3x3 grid):
  * Defensive third: x < 33 (left: y < 33, center: 33 <= y <= 66, right: y > 66)
  * Middle third: 33 <= x <= 66 (left: y < 33, center: 33 <= y <= 66, right: y > 66)
  * Attacking third: x > 66 (left: y < 33, center: 33 <= y <= 66, right: y > 66)
- Event locations (x, y) are in the team-in-possession frame; opponent-actor events are flipped into that frame
- The textual field pitch_zone (and pass_end_pitch_zone) uses the same 3×3 rules: for players on the team in possession it matches that frame; for opponent players it describes the zone from that opponent's attacking perspective (so you can read where th

In [21]:
import os
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

from possession_analyzer import analyze_possession  # o come lo importi tu
from possession_description import generate_possession_description

vertexai.init(
    project=os.environ["GCP_PROJECT_ID"],
    location=os.environ["VERTEX_LOCATION"],
)

model = GenerativeModel(os.environ["VERTEX_MODEL_NAME"])
cfg = GenerationConfig(
        temperature=0.4,
        max_output_tokens=2048,
    )
# possession_events: lista eventi Wyscout ordinati per quel possesso
# match_info: output get_match_details (con squadre, ecc.)
# all_match_events: tutti gli eventi partita se vuoi preceding + match_state
analysis = analyze_possession(
    possession_events,
    match_info,
    all_match_events=events,
)

text = generate_possession_description(
    model,
    analysis,
    generation_config=cfg
)
print(text)

e:\Edoardo\Education\SportDataCampus\ScoutingAgent\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


SECTION 1 - GENERAL POSSESSION ANALYSIS:
This brief possession sequence for Real Madrid began immediately following a Celta de Vigo attacking move that culminated in a long, progressive pass from Marcos Alonso to Ferrán Jutglà in the attacking third, right zone. However, this Celta attack was swiftly cut short as F. Mendy of Real Madrid initiated a counter-pressing recovery duel in the defensive third, left zone, at the exact location where the ball had landed. Almost instantaneously, Ferrán Jutglà was involved in a loose ball duel at the same spot, indicating a rapid transition and a contest for possession. F. Mendy then executed a clearance, specifically a head pass, from the defensive third, left zone, pushing the ball slightly further back into the defensive third, still in the left zone. The possession concluded shortly after with a game interruption due to the ball going out of play, originating from the defensive third, left zone. Throughout this very short possession, Real Madr

In [19]:
from possession_description import (
    render_player_section_possession_prompt,
    build_target_player_events_text,
)

# analysis = analyze_possession(possession_events, match_info, all_match_events=events)

general_text = "..."  # output reale Section 1, oppure "PLACEHOLDER general analysis"
pid = 276979  # Wyscout player id

print(build_target_player_events_text(analysis, pid))  # JSON eventi del giocatore
print(
    render_player_section_possession_prompt(
        analysis,
        general_description=general_text,
        target_player_id=pid,
        # target_player_name="F. Mendy",  # opzionale
    )
)

[
  {
    "phase": "possession",
    "id": 2927832885,
    "minute": 0,
    "second": 7,
    "matchTimestamp": "00:00:07.268",
    "matchPeriod": "1H",
    "type_primary": "duel",
    "type_secondary": [
      "loose_ball_duel",
      "recovery",
      "counterpressing_recovery"
    ],
    "teamId": 675,
    "playerId": 276979,
    "playerName": "F. Mendy",
    "location": {
      "x": 30,
      "y": 3
    },
    "pitch_zone": "Defensive third, left"
  },
  {
    "phase": "possession",
    "id": 2927832886,
    "minute": 0,
    "second": 8,
    "matchTimestamp": "00:00:08.091",
    "matchPeriod": "1H",
    "type_primary": "clearance",
    "type_secondary": [
      "head_pass"
    ],
    "teamId": 675,
    "playerId": 276979,
    "playerName": "F. Mendy",
    "location": {
      "x": 31,
      "y": 3
    },
    "pitch_zone": "Defensive third, left"
  }
]
You are a football scout of an elite football team. You have the same structured match data as for the general possession analysis, pl

In [22]:
from possession_description import generate_possession_player_section

text = generate_possession_player_section(
    model,
    analysis,
    general_description=general_text,  # deve essere il testo Section 1
    target_player_id=pid,
    generation_config=cfg,
)
print(text)

SECTION 2 - TARGET PLAYER IMPACT ANALYSIS:
F. Mendy's impact in this brief possession was defensive, immediately engaging in a loose ball duel and counterpressing recovery in the defensive third, left zone (x=30, y=3) just as the opposition's attack reached its conclusion. This action was crucial in regaining possession for Real Madrid. Following the recovery, Mendy executed a headed clearance from a similar position (x=31, y=3), preventing further immediate danger and ensuring the ball was moved out of the immediate defensive area. While this possession was short-lived and primarily defensive for Mendy, his actions demonstrate a proactive approach to winning the ball back and securing possession, contributing to the team's defensive stability from deep within their own half. His positioning and immediate engagement highlight his defensive responsibility and effectiveness in transition moments.


In [ ]:
from possession_description import generate_possession_descriptions_pipeline

rows = generate_possession_descriptions_pipeline(model, analysis, generation_config=cfg)
for r in rows:
    if r["description_type"] == "player":
        print(r["player_id"], r["player_name"], len(r["description"]))

276979 F. Mendy 1027
569212 Ferrán Jutglà 1126


In [25]:
for r in rows:
    if r["description_type"] == "player":
        print(r["player_id"], r["player_name"])
        print(r["description"])
        print("\n\n\n")

276979 F. Mendy
SECTION 2 - TARGET PLAYER IMPACT ANALYSIS:
F. Mendy's impact during this brief possession was primarily defensive, stemming from a Celta de Vigo attacking sequence that transitioned into a turnover. Immediately following Marcos Alonso's long pass, which was intercepted by Ferrán Jutglà in the attacking third, right zone, F. Mendy was positioned in the defensive third, left zone (x=30, y=3) and engaged in a counter-pressing duel. His decisive action was to win this loose ball duel and immediately execute a head clearance, moving the ball slightly further into the defensive third, left zone (x=31, y=3). This action prevented any immediate counter-attack threat from Celta de Vigo and ensured the ball went out of play, effectively ending the very short possession for Real Madrid. His positioning and immediate defensive action were crucial in mitigating the risk posed by the turnover in a dangerous area, although the overall possession was too short to assess any offensive c

In [26]:
rows

[{'possession_id': 2927832885,
  'team_in_possession': 675,
  'team_in_possession_name': 'Real Madrid',
  'opponent_team_id': 692,
  'opponent_team_name': 'Celta de Vigo',
  'num_events': 4,
  'temporal_moment': {'period': '1H', 'matchTimestamp': '00:00:07.268'},
  'description_type': 'general',
  'player_id': None,
  'player_name': None,
  'description': "SECTION 1 - GENERAL POSSESSION ANALYSIS:\nThis possession sequence began immediately after Celta de Vigo lost possession in their own middle third. Initially, Ilaix Moriba played a short back pass to Marcos Alonso in the center of the middle third, who then attempted a long, progressive forward pass towards the attacking third, right zone. This pass was intercepted by Ferrán Jutglà in the attacking third, right zone, but the ball was immediately contested. F. Mendy of Real Madrid was involved in a counter-pressing recovery duel in the defensive third, left zone, at the same location where Jutglà was also involved in a loose ball duel